# Dynamic programming, Part 1 of 2: iterative policy evaluation

Part 1 of a two-part dynamic programming problem (Part 2 is **u20-p03**). Evaluate a fixed policy on a known MDP by repeating Bellman expectation backups, as in u20-l03.

```python
def policy_evaluation(P, R, pi, gamma, theta, max_sweeps):
    ...
    return V, sweeps
```

- `P`: float array `(S, A, S)`, `P[s, a, s2]` is the probability of moving from `s` to `s2` under action `a`.
- `R`: float array `(S, A)`, the expected reward for taking `a` in `s`.
- `pi`: float array `(S, A)`, the policy's action probabilities (each row sums to 1).
- `gamma`: the discount factor, $0 \le \gamma \le 1$. `theta`: the stopping threshold. `max_sweeps`: a positive int.

Terminal states are stored as absorbing states with reward 0, so they need no special handling.

Start from $V_0 = 0$. Sweep $k = 1, 2, \dots$ computes every state's new value from the **previous** sweep's values (a synchronous sweep with two arrays, not in place):

$$V_k(s) = \sum_a \pi(a \mid s)\Big(R(s, a) + \gamma \sum_{s'} P(s, a, s')\, V_{k-1}(s')\Big).$$

Stop after the first sweep $k$ with $\max_s |V_k(s) - V_{k-1}(s)| < \theta$, or after sweep `max_sweeps`, whichever comes first. Return `V`, the float array $V_k$ of shape `(S,)`, and `sweeps`, the int $k$. Do not modify the inputs.

**Example.** The 3-state MDP from u20-l02:

```python
P = np.zeros((3, 2, 3))
P[0, 0, 0] = 1.0
P[0, 1, 1] = 1.0
P[1, 0, 1], P[1, 0, 0] = 0.8, 0.2
P[1, 1, 2] = 1.0
P[2, :, 2] = 1.0
R = np.array([[1.0, 0.0], [2.0, 0.0], [0.0, 0.5]])
pi = np.full((3, 2), 0.5)
```

With `gamma = 0.9` and `theta = 1e-3`, `policy_evaluation(P, R, pi, 0.9, 1e-3, 1000)` returns `V` close to `[4.0887, 3.8880, 2.4915]` and `sweeps = 54`. With `max_sweeps = 2` it returns `[1.175, 1.5175, 0.475]` and `2` (sweep 1 gives `[0.5, 1.0, 0.25]`).

The exact $v_\pi$ is `[4.0971, 3.8965, 2.5]`: stopping when the last change is below $10^{-3}$ still leaves an error near $10^{-2}$, because the remaining changes add up to about $\theta\gamma/(1 - \gamma)$.

**Restrictions.** Only `numpy`. No `np.linalg` (no `solve`, `inv`, `pinv`, `lstsq`): iterate.

**Constraints.** $S \le 300$, $A \le 5$. Time limit 1 second for $S = 300$, $A = 5$, $\gamma = 0.95$, $\theta = 10^{-8}$: vectorize each sweep (`P @ V` has shape `(S, A)`).


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem2_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem2_submission.py
import numpy as np


def policy_evaluation(P, R, pi, gamma, theta, max_sweeps):
    # Your code
    return V, sweeps


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem2_submission as submission
importlib.reload(submission)
# e.g. submission.policy_evaluation(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwtz2ziS/itYpWpNJjIjO8nUnBLnzpl4ZrOVhyvOevdKdnEgEZK4okgOQcX2eP3f7+sGQIJ6OI/ZvcyURYJAd6Nf6G4At7006Q1Fb3U42C8Hh72+6C1VLdF0u/mlTutMUdvrm1wu04koq2JWyeUyzWd9cSqrWhyIYioOhyKtVSXr9LMSZZGlkxuhPstshZYiJ0iZ0hpPFnw2eMKI03xVK43Wg0O8Jul0mk5WWX2DFmpQeV3Rc8+AjLsgl0WyylQMwphEkDbO1PIw1qvxMtXa9pJZVlypJE6XZVHVhGvUy1fL8qZ3ia9jmecq4cYszWU2oyG6yD4rekjzz/RT2t9M1/o3NyzOiqIkcFOZaUW8UrqO63SpilUd04fDAbXKmcFZZVFimBh7TCSw+LJ1fmgfqyxbSn5jmqOlrJerjGnQxaqa8MyLKp0R8UwrZHKIxrpaqTu8T9OMGXzbm8sqhxCikjl6wf/9RWWlqrSo57IWloECtCXUaDgmgl+nVbEUdrhrffhrGF3kF/mx7S5SLSZFXkNkj1d5WuvHL+jn5WMLFe9lmrx8bHqDiEi8qUWipimgCuKdmK7yCc1dCxJpwo3xQ0NbLRcKT+oib6UrjAKIX9H0ayRO/nH87vTtyZnIUsiJOhuwBEwLPS+ucoFBH1e5CD6nOgVV4XNxRuBqUa2AVn1W1Q0PwtR+lmm2qjCykqlWgl7FVVrPhRSAp+VMiasqrTFhMS0q8UrlQ6Hljbgicq+kxteCjCQnqCKB9MQ8TRJ0T/NyVWvinpFCblk6Kcqb5oUUifhrX1n6AlDzklofiJ+KTI7FT6d/00JWSmhS8op4lRMpe1q8OxSnVdEXuhBZMZEZQYTCabFcgSljRUwFiwoyuRSfs5T4MMb85xDlVE7qoooI02E0IOFKMVuR/AODWQNWpvbreaVkInSpIDD0qorVbJ7dQFuyqQAtoGJSVMToShHTa5FOiUC8sjnaeVizIaacnZ6cvI5/Pv7p04eP4oiw04Qv8kkmgZ3EEJxcT1RJmhIOL3KBfyU+mV5QKHBcJQF0MemLpZ65PsCbF8TkPLEt9K8VbkB9WyBxXgbXbiy8kDfGSaioJnOvdYr5p7muZT5RwXXffI8+qVwXVegNZ7SqXlW5uI4SuN7JPAijSbnCXxZzEJrOiqcp3jC6k6oqKg+KmbIHKy8jqWVVyRvQ3c5Dz2WpgllR96GUOf2Ffh5d9KDZcLQXPUeZBq/rVZmpgGaO/mHEQy0tzFP0cZ0IVtgX04veLQG8g8C1wSVuNfyOui7VpIZO3Hr974CupWySFfo+yvqiqovs6EDt/9AX0j7+6OiFZV2B5IZYqesbTHSaFZIIo3bG2f3gTWZm5kdTujKPO6ZjJmC7h9vm5j7x9BoEEEiq4d6wMAYz0JFlQSjgKUgL/W9X9puPnjypTNkT5jQmzacNcKvIJO8ss0wEMyy76I9lF/3xFU9VFTjGjCDilvKafuRYBzOxL67CMCTYmGj6uxIKi5oYkPFtMZUun9jRCVq64C2DTFYzcruKFFbc4mcYPZndhR3Zq99WMvsKrewalelKROcJa3o/ZI6CCFnXlf1+0WNxtEC2qIunIG2f/7ha+KrB9MeWDyBuw5h8rjYQSCrDdYqhDEQtT76FAWd8i09/qjqkUS80dYSR6phNhMVBT3qnPBihJ5F2yAb91ApWWMBhFHOgFscdcmZwfyQ5i3XPddpzULuUxjqXJVby+v/XNWdFrr7RKQOl08prUkle23shlsSErbdLT5CkEzCSApfQp6c7uQ51BDBIlCqPPiHS8xTMEvgJ7FsnbxNA2FlCqCkimPztusN4iCWQfTH+fr7Lr+K76WLsgvF9L9slsZ3h6K4rsIikQaKDcbjN12B462e2Dm/8xNg+smy7lk0z6BtvB83OfVk5OAzBC16KeLmqOQcIpnlfPIQ7hWU8fLi4oidHiQkdf8IKIKY5Y55SgEpzQIyDKHAG85I5QsZqtloiKMcSIspMTlTkok5e9cme4BdHrWXJkONZSQMI46XpieiMFo88WCfIfKYhKRQZE9YhDVUIZSghVMHvaRmYMYyjo+DsTxrlsk7kBokNdDFRFOCn05QiSwqZ27k8uE3FI3Fw91xcFdWConrJyitIfIhHG39pmQziWw5TEJ/mgVYUCoKqr+Xyc5/F8K5+yL0lkua8E+uhRuyNGRCDONbda0YieDc5FKH6ABCVS681TYnj8HdyIiZAQEMwSRN1U7bCgT0FB8gu9serBJ6UmjkGR1ieUgK1KmFOor5KJ4oAgkMpVuhiai0YNDuOalYhapkiXqLlmyBgxArfIvGJPqQVmjGZbI9Q0LpgkBGHVdJRq3pA0SQgRMgupzGDUZXzNl9QpaTePhgxSm3DETvfI2GFKB4LP21oVTImVazIGoJDX+8gRKB5cWQhrbmiMXKaRdskD/ticQhsXf/IJPfXGjsz+TIrmE5ixCGz4dBrZjYgYwsSWmJ3sMOPN/0JsR3VRcF6g+Q/qYfR4fQOUT60TD23WVejxKxMMDsGJFY55fO3ljV2HOeQSOyciLf8Y5TILJEGT6FaMwpgTZJ5a+XEgMg67jXQKp/BOlXiBNYmOJBkUizB7amE+sVNxwuukKy0lGkRc4WhU+X4hVpgAaCsLR6k7DTmaZaQ1U2oMoKEOSVboJ5/PfvwHrTmbDm6TkAh22p5U8/Rsv9GODzihS1xxElavRQvWhz4iHdY5b+4rX65kfLD2Lz0H4K9rrN03DSZH7REqzrN2uaiefynLtoiQSnruT9cp7NctsP0jV4rL7iXCuvCWE4WTQu0OKdigak16EmKaeaqcXdoWKT1fqbQTRxEP0QHNqbTQ6oTLanY12eXguzCLj7O+ZUyBZdnF7nDEU3TDBrtXoOLHsguKkUZoK20IBKNHv4tX+RUw+HqXCUKrgAAYVqS1KKHNlL8y8nHE1iO5UV0it8gjqkMFsdhBMdFwxHVlbICZRc5uBJR5wjaqqo6GMAYED4QmBAAt9W+uBYkHmCt/k0OxcnTwSFXwt7+/fh/z+JXx+/fn7wGBbcXvdhWHeOYJoOJOXmaV3WtJvYJcb55mhTLErRe9O4u8rcfPpzG7z+8PjkDuAC6Ev1cwFbp4e9QXGUej/VNPmk+vEUM+RNgmLcz5b28RpDZvv2iclqfi+rkugz9EssnU4v5UpUllsj0lgHp2GoJf4NlSzUWy6miA+RXIyiymcSTuZosdKCrCYm4lt319qOxd4qGqbYMPhYLw5ylnoExz+EWKQI0NT42G13vNdLhYMiWEmmhqqvUlBU7S5RZhEaX26LZulII7JhL0BLk1yC0G4ee3SA/v+Y4lIpyajM+HFmyh9AWrg9b6ofkJjUPtylykRs/c6si+s2LuyE9ozeSnztLoK1i05IAfkVwzLCTtdI28Wh0aQkdy6ad18naLTlcDo1XmoF57WPYQ2xqpbadVBwhI2w3TWAqFIlVJlAwgAHB9A8oo6GgOx1TOR/TWXDsprKIFNsi4Hq5x3NTyvZbplwdgasm3wzmX8lsEZAw1pZuL0yn7kafTWqwnlU0oSx1jJjc4eYCZksqiJcDyZ0oKF9SacG8RsAGRlLkGCwRecIdUTJHwJdMrBFEONy+NnqyiGSSWBzhZudWNl6/SJdZSuImnKP9g8stA1vhRasyoah7y1g/SczuZ+PP8Hrrs4FAISvmoy28Y/qtQW3hJY+wjMTzH+DiOgcBbY0LHus6HNimAWtDdzHvu4byPL+V6cfOdtbnvqYQTAcZ2ldBfe955PsApsnuWW3vtA1ju1it42TDj2RZqjwJXMXHDLroGZdn3Pt/d/m1DQuvgq/S/INdxY5Xs2NNa1BoaitrA6LC9nsn63cIGTd02vigbyQOhHkK6YGkINGC2un+XRRgrMSoPLWyQVC/6J8FpafsYAMPTxhu9y6PaEUBYxyo272+2DNAnEWRAe4BH/Le2R6VYu+21xO30U8liy3ka0d7C+xBs4M4lrkm3k7mdoXgULBdILj8rCSS2H/wb/jcZKPZlbzR+2ZPFpkM4sK0AURKaaEVeXbTdziRubRLTWC0wBSRbChFQceNyNQMqfkVhU28Mx1EURR6ixmCMpEUJBQDeJ5yfmmkEHiL5p95cfRWYUOuXXyhhv8SgbeW/ll04kJPhYDhm1UHFGhh+UPwAx1u0R1ADv+IhPPC4Wi2ZFspg/Lu5GNjRHFRmniyW22j7sbI1kzvayZMGSX04n+Eg00pq2c3IjCRk8UwGlz6yr1l+t/CghbrfZNn57YxZW79vgmbMImVvTNRNJMZk1JXaq5yTmKb2JFH/Xvnb2d2d0+mPq1SgMluYo5kA4BAohtTNtWN6D/kap/pTJSeVClnFaKEqsK+Z8LuRHOH1G1fU8IGc1XUT/NmfbfCNKayistbI+TNeK7jehyoKI6bD0j5bLBJwBFpTnlRnxIegICkppR+KhfzUebXTMFGpVdUMqGCFXlZrk1apcNfCse8qL3TRKJghaHevJ3Wkm+ZSTsmvEOivP0Riv7vbhnrHfV3zF4iEg+aXAyedj3LpUxWVrPPoyZEdFPZ2fPw0kuJLacoAD8STZcnLlyH3qOdCg5RVshEBwET8ZjUhmyCSxE9goc1iyoZgXN2SJ8IoiXG7+C4wdXEI7FpFyZhpKbRJb1TqqfbV1OYMSprM2nSRGp4X0AEPvxRC+2SyNmRijaGThEsmaAeGaouebWgN7aSSz+cdSmdi2c11x030PpegqsWAbOTDofowHT3V3mjJa7kOQXRaRGd1VQ+efPBMc8UeSLzE9i3sze/HL/9+K5vs3TbFe4s5uKu2wX2fFn39BLx8nDgaOlmx3wIpi1WQZxJWikYnxFGAErXwyxHo6pTQlE5Mt98evPu5GP88eT4bb+hbi3M0aUi7emWwiJqjak2Yyo7VJ3kLRNvRt7xMKMabWEOYvJc1Rq+1XgTnYXFCAl1QH/WB8JiTD89os+cMFwaxd+cEtuQqiKqAcVmGHme8DsYN+jWJmzZpVOSsIpojePSeDMzR/LA7NXY49ZFseA9BXHr5DGMBlQBdq7LYnkltWoqRKYG0pbEXr09GQwOWgK+aRq7KLYbQ0wm7cTQCbHCCsoGQfesR41Zb4DexijnVwgv9Jk0infiAyTgTweDwfDyuw2ZzzR26zjc5J/vMH3cXrHb0cRc+uZT+GXhtrxKCmWCJnPYDqsLgbjz8/ZvnMLs22zyomcO/ZEVNktGcw6wmbc9R/hl25t5xjfbZUsGmu1lj/vZwpbL8kwX8gzu6KAL4q3bo7WGgoYAicrUOHZ6It/+WVba4YjSWi11wKdn8u6qwF7V7fvTBooEGcE09Ate8BK0CckhCWFcC555OSbtr1bwW0YpKFrhUwS5m9qawyX3nOYr5ctvYhZY4w+HDGT0bHgJ/837wqDWVqkFu8eLngXtOnvY6CtVgynt2ZamWH9rNn2oaRAN7r5lT2zz4MN3rDp/dOWxm3ObXvm7XJoVwqjlHFkqBd6bZzc2PHg72MQexsbZWWdFPnP7Mus+WwRpbk6ZcTTfpwRiKivhdgZD3wlY5Ly5sVbT/oMTbmim8HrLWZVvWkv+XbT45wx4vyL5yjWkhdUoOMGroMVJsGujti+ebFvebDh76RKyqnFs37wUNQNMrHpEqWPQxq73RaXsVMZFkQUbhJlvBKruaC6BqzvgmhEuQdm5pPCx5qlwCY/xbnFM+Q1tlA3d9gClO2Znt1ksaE9324noXbt0fd7R7puzpn3/cE3fngPp03YzUdQcH4fDt45bXctlSQ7w0ju5aNriZVI2ydgp1cXL6HdVFciKnvTp7sITt4qcjiB8+p9kcuAOVHLrAf5faz3gvgdIbtwzjxtEP+IpOvS64f/DtcFAO1xv/WhoMweDRyO0E6ABEIwOveeBeX52edlN90/74iOfhJquoAM8t5A7ept6ZleeOaKZ22d9cYxB8ETVku4lHA2aU7tkKO2uvWPeLIIzTyfzTPHJVMZ1RmieUJye/q6O6PU4DD1zfiACg+cMfSo6aqNXS4oID9qJz6K8qJacFrUwrH6mm3iRMerg2KE8C7cUWy1W79STbo+YnMHU21kTZcOtByUe0LZ4UY0p9Ha9OR9FoMbrXKWuZJUI7/Dt6UgbPei2scB1R+A897XeHWGWaSu6WZUmV0WVJY0u50Z8GP0UOvZDXzx1ZYHPygRD+6SXYBLtlB/Qr/e+f+BCG8LVMQsnrNBTJ2YljYoGOzjqr+m8HUOHJRGzJZO1U2ZM33oEAC/DZ9YPnm0JDZh9dOoM+v+Rny0f+102746pmLVwLhiWpJ8RqQWAkq8tOw4NHeWhI9cV1hiaAbEsh8IchOIhHh41HSbUYeJ36Mh3u2FaTsJFdCxTTVWlaDPEib4vZnK5pKOYc0TBoEpex/pKqbLhHTnAU3OicTSwwjz3JXlmJ7ggDXMlMko0FuKFB8/j+EI8OhLeEp5f1wagSnMYLVVPZN2v919qSZEjaD33mHge5+pqo39fS/TnlCIlVXtkJka8vPYDOXMW0gz3TrwboPtA5GNCN/7QicAthBeGZbuPilnBnPfFwrvkQOUlk7mZw27IAFTWXm34r/Zqw0HjJc3ZBP8kKABs3hGzxxbD7UezAaRvLmuYNTxTObWFZA+H5rA203I3bI+icncRYApWJ5rkjGYFmrj662mEu3Hi5EXHy+Nzc6g/pgGtBm7Qeu6dtOdh/nF7R9q5f+Te9tg40O6Grl3AoABkwfNddIGaybmLUHSKEl3VjHJBOkx/u3An5Hdct3ATPP/ynYs1u6OwDGwYDZ9edv3aYkHuLLB826fV3T4/ggPYdGvo/xI21Zw59ki7x+h5lV8swhGt+V8ifce9D8dEjp7PaUOGive2hEvHdRaLO8vhDTnhS6DrohRyWnPO4o6V3nOq0J4sZIhwNJikcFdMrGWCBJBUXDmfRjxxp8npMh4P3dP2LkXYuY6xIdCO8jWXkNbZ1J4YNzr2J6vDi+HO+zIe3+zlN3FrRt/LLk4j7M1E7tYHzkm2Sjh4mKtdrKOdJEkHqHJlOGLuIVLGBfie2+9csPDDXk58h+1q3jcxUycANl89J+fp3CCCc4NfQ8x4gDwlNCfebES9jpPuESpLUkz7uzuwe7EmTOSHNq36ap+5RiCbxJPv86F0P+aLntPcy4k7vvA+GoxGer4VCGGvobspddE7t+YDZk7q7EY8cQ4tUHIyNy/GudFm4apWTn0owhScKt2jOJv/kPfR5c1ipRtTOu+zU2RY9uKXzGhf50aYczAJl7PmdBZ4qcwwsKTvY9y+BK7pRStv7WsEu8y+ywCsd8vrtZhwFDzjXMwwmUNVvD3lhmcmkwmeEOv7JsF6Fl5u891rajeAS140SU431thmB5Y6urxoLGFgrjnY28DvXp/CZ/J1hk2zaOLzmKHEBzvMwovj7zNJDm5ByNOWkIve0+un7fi+naq9xN8uWwebxLVOJJ7I8ssGS/fhf/QUfZfXMG6DRIVgluuKDSK6EPwMhgQ9pxMiCnrAJ0Ch3zOyrE0i6T7Dl0ljLRjgzzPiTMel2OsizLqvcSnPuqL+zziW3t3d/wHoDrsC")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
